# 🎓 Ciência de Dados — Aula 10: Preparação Preditiva e Regressão Linear (Simples e Múltipla)
**Professor:** Dr. Rafael Duarte  
**Formato da Aula:** *Live Coding & Literate Programming* (100% guiada via Notebook)

---

### 🗺️ Onde Estamos no Semestre e Roteiro da Aula 10
* **O que vimos até aqui:**
  * *Aulas Anteriores:* Estatística Descritiva, Covariância e Correlação Linear ($r$).
  * *Aula 09 (Semana Passada):* Frameworks de Resolução de Problemas (CRISP-DM / KDD), Qualidade de Dados, *Data Wrangling* e ETL + Apresentação Inicial do Projeto.
* **O que faremos HOJE (Aula 10):**
  * **Módulo 1:** Da Correlação Descritiva ao Aprendizado Preditivo Supervisionado ($X \to y$) + Apresentação do Caso Prático de 20 Franquias.
  * **Módulo 2:** Preparação de Dados para Modelos Preditivos — Reta de Mínimos Quadrados (OLS) vs. *Outliers*, *One-Hot Encoding* (`get_dummies`), Separação Treino/Teste (`train_test_split`) e Imputação sem *Data Leakage*.
  * **Módulo 3:** Regressão Linear Simples e Múltipla Passo a Passo — Geometria dos Resíduos ($e_i = y_i - \hat{y}_i$), Interpretação de Pesos ($\beta_i$) e Auditoria das Métricas ($MAE$, $MSE$, $RMSE$, $R^2$).
  * **☕ Intervalo (15 min)**
  * **Módulo 4:** Governança Definitiva do Projeto do 2º Bimestre (Alinhamento Pós-Aula 09, Checkpoints, Multiplicador $1,2\times$, Avaliação por Pares de Soma Zero e Registro de Grupos).
  * **Módulo 5:** Laboratório de Fixação Acumulada (Aplicando o *Data Wrangling* da Aula 09 + *Regressão Linear* da Aula 10 na Base Própria ou no Dataset de Fallback `dataset_fallback_logitech_frete.csv`).


---
# 📘 MÓDULO 1: Da Correlação Descritiva ao Aprendizado Preditivo Supervisionado

## 1.1. A Grande Virada de Chave: Descrever o Passado vs. Prever o Futuro
Nas aulas do 1º Bimestre, estudamos **Covariância** e **Correlação Linear de Pearson ($r$)**. Na **Aula 09 (semana passada)**, aprendemos a estruturar problemas de dados e limpar bases sujas (*Data Wrangling* / ETL).  

Agora entramos oficialmente no território do **Aprendizado de Máquina Supervisionado (*Supervised Machine Learning*)**. Mas qual é a diferença real entre calcular uma **Correlação** e treinar uma **Regressão Linear**?

| Dimensão | 🔍 Correlação de Pearson ($r$) | 🎯 Regressão Linear Preditiva |
| :--- | :--- | :--- |
| **Objetivo** | Medir a força e direção da associação linear entre duas variáveis já observadas. | Construir uma **função matemática parametrizada** capaz de estimar valores futuros ou inéditos. |
| **Simetria** | **Simétrica ($r_{XY} = r_{YX}$):** A correlação entre `Marketing` e `Vendas` é idêntica à correlação entre `Vendas` e `Marketing`. | **Assimétrica e Direcional ($X \to y$):** Definimos claramente quem explica ($X$, *Features*) e quem é previsto ($y$, *Target*). |
| **Escopo** | Apenas pares de variáveis ($X$ e $Y$ isolados). | Múltiplas variáveis simultâneas ($X_1, X_2, \dots, X_p$) atuando juntas sobre o alvo $y$. |
| **Entregável** | Um único número adimensional entre $-1$ e $+1$. | Uma máquina preditiva ($\hat{y} = \beta_0 + \sum \beta_j x_j$) que recebe novos dados e devolve previsões em R$, Kg, minutos, etc. |

> 🏛️ **Contexto Histórico & Etimológico:**  
> O motor matemático que encontra a melhor reta entre pontos — o **Método dos Mínimos Quadrados Ordinários (*OLS - Ordinary Least Squares*)** — foi desenvolvido no início do século XIX por **Adrien-Marie Legendre (1805)** e **Carl Friedrich Gauss (1809)** na astronomia, para prever a órbita de corpos celestes (como o asteroide Ceres) a partir de observações telescópicas ruidosas.  
> Já o batismo do termo **"Regressão"** veio décadas depois com **Sir Francis Galton (1886)** ao estudar a hereditariedade da altura humana: ele observou que filhos de pais extremamente altos tendiam a "regredir" em direção à média da população (*"Regression towards mediocrity"*).


## 1.2. A Anatomia Matemática da Regressão Linear e os Mínimos Quadrados

No **Aprendizado Supervisionado**, fornecemos ao algoritmo uma tabela histórica onde já conhecemos a resposta correta ($y$, o "supervisor" ou gabarito).

1. **Regressão Linear Simples (1 variável explicativa $x_1$):** Ajusta uma **reta** no plano 2D:
   $$\hat{y}_i = \beta_0 + \beta_1 x_{i1}$$

2. **Regressão Linear Múltipla ($p$ variáveis explicativas $x_1, x_2, \dots, x_p$):** Ajusta um **hiperplano** multidimensional:
   $$\hat{y}_i = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \dots + \beta_p x_{ip}$$

### Dicionário de Sobrevivência (Quem é quem na equação):
* **$y_i$ (Valor Real / *Ground Truth*):** O valor verdadeiro que aconteceu na linha $i$ da base histórica.
* **$\hat{y}_i$ (*y-chapéu* — Valor Previsto):** A estimativa calculada pelo modelo para a linha $i$.
* **$\beta_0$ (Intercepto ou *Bias*):** O valor base de $\hat{y}$ quando todas as variáveis explicativas $X$ são iguais a zero.
* **$\beta_1, \dots, \beta_p$ (Coeficientes Angulares ou *Pesos*):** Representam o **efeito marginal (*ceteris paribus*)** de cada variável. Ex: *"Mantendo todas as outras variáveis constantes, cada $+1$ unidade em $x_1$ altera a previsão $\hat{y}$ em $\beta_1$ unidades"*.
* **$e_i = y_i - \hat{y}_i$ (Resíduo / Erro Individual):** A distância vertical entre o ponto real observado e a reta do modelo.

### Como o método `.fit()` descobre os melhores $\beta$?
Ele procura os valores de $\beta_0, \beta_1, \dots, \beta_p$ que tornam mínima a **Soma dos Quadrados dos Resíduos ($RSS$ — *Residual Sum of Squares*)**:

$$RSS = \sum_{i=1}^{n} (e_i)^2 = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

> 💡 **Por que elevamos o erro ao quadrado $(y_i - \hat{y}_i)^2$?**  
> 1. **Evitar anulação de sinais:** Se somássemos os resíduos simples $\sum (y_i - \hat{y}_i)$, um erro de $+100$ e outro de $-100$ somariam $0$, fingindo que uma reta péssima é perfeita.  
> 2. **Geometria e Derivada Única:** A função quadrática é uma parábola suave (convexa) com um único ponto de mínimo global exato.  
> 3. **Penalização quadrática de grandes erros:** Errar por $2$ custa $2^2 = 4$; errar por $20$ custa $20^2 = 400$!


## 1.3. Nosso Caso Prático da Aula 10: Expansão da Rede "Café com Dados" (20 Franquias)

Seguindo a mesma filosofia visual da **Aula 09**, não vamos usar uma base abstrata de milhões de linhas onde ninguém enxerga o que está acontecendo. Vamos trabalhar com uma base enxuta e rastreável de **20 franquias (`Loja_01` a `Loja_20`)**.

**O Problema de Negócio:**  
A diretoria quer prever o **Faturamento Mensal (`Faturamento_Mensal_Mil_BRL`, nosso alvo $y$)** de novos pontos comerciais a partir de 4 características ($X$):
* `Area_m2`: Área útil da cafeteria em $m^2$.
* `Fluxo_Diario_Pessoas`: Passagem média diária de pedestres na frente da loja.
* `Investimento_Mkt_Mil_BRL`: Verba mensal de anúncios locais (em R$ mil).
* `Tipo_Ponto`: Categoria do ponto comercial (`Galeria`, `Rua` ou `Shopping`).

**Conexão direta com a Aula 09 (*Data Wrangling*):**  
Reparem ao rodar a célula abaixo que a planilha enviada pelo comercial ainda traz algumas "cicatrizes" clássicas que vimos na Aula 09 (como valores **`NaN`** nas Lojas `05`, `12` e `14`, texto despadronizado e um *outlier* absurdo na `Loja_18`). Vamos ver como tudo isso impacta diretamente o **Aprendizado Preditivo**!


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações para exibir todas as 21 linhas na tela sem cortar nada
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 1000)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

# Base das 20 Franquias (+ 1 linha duplicada no final = 21 linhas)
dados_franquias = {
    'ID_Loja': [
        'Loja_01', 'Loja_02', 'Loja_03', 'Loja_04', 'Loja_05',
        'Loja_06', 'Loja_07', 'Loja_08', 'Loja_09', 'Loja_10',
        'Loja_11', 'Loja_12', 'Loja_13', 'Loja_14', 'Loja_15',
        'Loja_16', 'Loja_17', 'Loja_18', 'Loja_19', 'Loja_20',
        'Loja_20'  # <-- Duplicata de exportação (visto na Aula 09)
    ],
    'Area_m2': [
        30.0, 35.0, 40.0, 42.0, 45.0,
        50.0, 52.0, 55.0, 60.0, 65.0,
        70.0, np.nan, 78.0, 80.0, 85.0,  # <-- Olhem a Loja_12: Area_m2 = NaN!
        90.0, 95.0, 88.0, 110.0, 120.0,
        120.0
    ],
    'Fluxo_Diario_Pessoas': [
        400, 600, 1100, 550, 500,
        750, 1300, 700, -350, 850,       # <-- Olhem a Loja_09: Fluxo negativo (-350)!
        950, 1400, 1000, 1500, 1600,
        1150, 1750, 1280, 1850, 2000,
        2000
    ],
    'Investimento_Mkt_Mil_BRL': [
        2.0, 3.0, 6.0, 2.5, np.nan,      # <-- Olhem a Loja_05: Marketing = NaN!
        4.0, 7.5, 3.5, 3.0, 4.5,
        5.0, 8.0, 5.5, np.nan, 9.0,      # <-- Olhem a Loja_14: Marketing = NaN!
        6.0, 10.0, 6.8, 11.0, 12.0,
        12.0
    ],
    'Tipo_Ponto': [
        'Galeria', 'Rua', ' shopping ', 'Galeria', 'Galeria',
        'Rua', 'Shopping', 'Galeria', 'Galeria', 'Rua',
        'RUA', 'Shopping', 'Rua', 'Shopping', 'Shopping',
        'Rua', 'Shopping', 'Rua', 'Shopping', 'Shopping',
        'Shopping'
    ],
    'Faturamento_Mensal_Mil_BRL': [
        92.0, 141.0, 236.0, 118.0, 125.0,
        175.0, 272.0, 149.0, 152.0, 192.0,
        216.0, 305.0, 233.0, 298.0, 338.0,
        258.0, 366.0, 8500.0, 398.0, 425.0,  # <-- Olhem a Loja_18: 8500.0 (Outlier absurdo!)
        425.0
    ]
}

df_bruto = pd.DataFrame(dados_franquias)
print(f"Dimensões da tabela inicial: {df_bruto.shape[0]} linhas x {df_bruto.shape[1]} colunas")
display(df_bruto)


---
# 🧹 MÓDULO 2: Preparação de Dados para Modelos Preditivos (*Model Prep*)

Na **Aula 09**, aprendemos a limpar dados para deixá-los consistentes. Agora, na **Aula 10**, precisamos entender **como o algoritmo de Regressão Linear enxerga os dados**.

## 2.1. Por que o Scikit-Learn rejeita dados não preparados?
Para encontrar os pesos $\hat{\beta}$, o `LinearRegression()` executa álgebra matricial pura: $\hat{\beta} = (X^T X)^{-1} X^T y$.  
Veja na prática o que acontece quando tentamos alimentar o modelo com texto (`Tipo_Ponto`) ou com os valores `NaN` das Lojas `05`, `12` e `14`:


In [ ]:
from sklearn.linear_model import LinearRegression

# 1. Tentando treinar com uma coluna de texto ('Tipo_Ponto')
try:
    X_texto = df_bruto[['Area_m2', 'Tipo_Ponto']]
    y_bruto = df_bruto['Faturamento_Mensal_Mil_BRL']
    LinearRegression().fit(X_texto, y_bruto)
except ValueError as e1:
    print("❌ FALHA 1 — Matrizes não multiplicam strings de texto:")
    print(f"   Mensagem do Scikit-Learn: {e1}\n")

# 2. Tentando treinar apenas com números, mas mantendo os NaNs das Lojas 05, 12 e 14
try:
    X_nan = df_bruto[['Area_m2', 'Fluxo_Diario_Pessoas', 'Investimento_Mkt_Mil_BRL']]
    LinearRegression().fit(X_nan, y_bruto)
except ValueError as e2:
    print("❌ FALHA 2 — Matrizes com buracos (NaN) não podem ser invertidas:")
    print(f"   Mensagem do Scikit-Learn: {e2}")


---
## 2.2. Aplicando o Wrangling da Aula 09 + Visualizando o Sequestro da Reta pelo Outlier (`Loja_18`)

Vamos aplicar rapidamente a faxina que aprendemos na **Aula 09** (remover a duplicata da `Loja_20`, padronizar o texto de `Tipo_Ponto` e converter o fluxo negativo `-350` da `Loja_09` em `NaN`).  

Mas antes de remover a **`Loja_18` (`Faturamento = 8500.0`)**, vamos usar o gráfico abaixo para responder a uma pergunta crucial de **Aprendizado Preditivo**:  
> *O que um único Outlier faz com uma Regressão Linear treinada por Mínimos Quadrados?*


In [ ]:
# Aplicando a limpeza básica da Aula 09:
df_limpo = df_bruto.drop_duplicates(keep='first').reset_index(drop=True)
df_limpo['Tipo_Ponto'] = df_limpo['Tipo_Ponto'].str.strip().str.title()
df_limpo.loc[df_limpo['Fluxo_Diario_Pessoas'] < 0, 'Fluxo_Diario_Pessoas'] = np.nan

# Comparando a Regressão Linear COM a Loja_18 (8500.0) vs. SEM a Loja_18
df_teste_outlier = df_limpo.dropna(subset=['Area_m2']).copy()
df_sem_outlier = df_teste_outlier[df_teste_outlier['Faturamento_Mensal_Mil_BRL'] < 1000].copy()

reta_suja = LinearRegression().fit(df_teste_outlier[['Area_m2']], df_teste_outlier['Faturamento_Mensal_Mil_BRL'])
reta_limpa = LinearRegression().fit(df_sem_outlier[['Area_m2']], df_sem_outlier['Faturamento_Mensal_Mil_BRL'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Painel 1: Visão em escala total
axes[0].scatter(df_teste_outlier['Area_m2'], df_teste_outlier['Faturamento_Mensal_Mil_BRL'], color='dimgray', s=60)
axes[0].scatter([88.0], [8500.0], color='red', s=130, zorder=5, label='Loja_18 (Outlier: R$ 8.500 mil)')
axes[0].plot(df_teste_outlier['Area_m2'], reta_suja.predict(df_teste_outlier[['Area_m2']]), 'r--', lw=2.5, label='Reta com Outlier')
axes[0].plot(df_sem_outlier['Area_m2'], reta_limpa.predict(df_sem_outlier[['Area_m2']]), 'b-', lw=2.5, label='Reta Real (19 Lojas)')
axes[0].set_title('1. Escala Global: O erro ao quadrado da Loja_18 domina o modelo')
axes[0].set_xlabel('Área da Loja (m²)')
axes[0].set_ylabel('Faturamento Mensal (R$ Mil)')
axes[0].legend()

# Painel 2: Zoom nas 19 Lojas normais
axes[1].scatter(df_sem_outlier['Area_m2'], df_sem_outlier['Faturamento_Mensal_Mil_BRL'], color='navy', s=65, label='19 Lojas Normais')
for _, row in df_sem_outlier.iterrows():
    axes[1].annotate(row['ID_Loja'].replace('Loja_', 'L'), (row['Area_m2']+1, row['Faturamento_Mensal_Mil_BRL']-5), fontsize=8.5)
axes[1].plot(df_teste_outlier['Area_m2'], reta_suja.predict(df_teste_outlier[['Area_m2']]), 'r--', lw=2.5, label='Reta Sequestrada pelo Outlier')
axes[1].plot(df_sem_outlier['Area_m2'], reta_limpa.predict(df_sem_outlier[['Area_m2']]), 'b-', lw=2.5, label='Reta Limpa (Sem Loja_18)')
axes[1].set_ylim(50, 1100)
axes[1].set_title('2. Zoom nas 19 Lojas: A Reta Vermelha erra TODAS as lojas normais!')
axes[1].set_xlabel('Área da Loja (m²)')
axes[1].set_ylabel('Faturamento Mensal (R$ Mil)')
axes[1].legend()

plt.tight_layout()
plt.show()

# Removendo a Loja_18 corrompida para seguirmos com as 19 lojas válidas
df_19_lojas = df_limpo[df_limpo['Faturamento_Mensal_Mil_BRL'] < 1000].reset_index(drop=True)
print(f"✅ Outlier removido! Temos {len(df_19_lojas)} lojas válidas na base.")


---
## 2.3. Preparação Exclusiva de Modelagem: *One-Hot Encoding*, `train_test_split` e Anti-*Data Leakage*

Agora estamos com as 19 lojas válidas. Vamos inspecionar quais delas ainda possuem `NaN`:


In [ ]:
print("Lojas com valores ausentes (NaN) que precisam de tratamento pré-modelagem:")
display(df_19_lojas[df_19_lojas.isnull().any(axis=1)])


Para transformar essa tabela nas matrizes de treino e teste sem cometer erros metodológicos, executamos **3 passos fundamentais da Aula 10**:

### 1. *One-Hot Encoding* (`pd.get_dummies`) e a "Armadilha da Variável Dummy"
A coluna `Tipo_Ponto` possui 3 categorias (`Galeria`, `Rua`, `Shopping`).  
* Se criarmos um número sequencial (`Galeria = 1`, `Rua = 2`, `Shopping = 3`), a Regressão Linear achará que `Shopping` vale matematicamente o triplo de `Galeria` e que a distância entre eles é fixa!
* Por isso, criamos colunas binárias ($0$ ou $1$) chamadas *Dummies*.
* **Por que usamos `drop_first=True`?** Se sabemos se a loja é `Tipo_Ponto_Rua` ($0$ ou $1$) e `Tipo_Ponto_Shopping` ($0$ ou $1$), não precisamos da coluna `Tipo_Ponto_Galeria` (pois quando ambas são $0$, a loja **obrigatoriamente** é de `Galeria`). Assim evitamos colinearidade perfeita na matriz $X$!

### 2. Divisão Treino vs. Teste (`train_test_split`)
Um modelo preditivo nunca pode ser avaliado nos mesmos dados em que decorou a fórmula. Dividiremos nossas 19 lojas em:
* **15 Lojas de Treino (`X_train`, `y_train`):** Onde o algoritmo aprende os pesos $\beta$.
* **4 Lojas de Teste (`X_test`, `y_test`):** Escondidas do modelo durante o `.fit()`, usadas para medir o erro real de generalização.

### 3. Imputação de `NaN` Livre de *Data Leakage* (Vazamento de Dados)
Se preenchermos os `NaN` das Lojas `05`, `09`, `12` e `14` usando a mediana das **19 lojas juntas** antes de separar Treino e Teste, informações do Teste vazarão para o Treino.  
**O protocolo correto:** Dividimos Treino e Teste primeiro $\to$ Calculamos a mediana **apenas nas 15 lojas de Treino** $\to$ Preenchemos os `NaN` usando essa mediana!


In [ ]:
from sklearn.model_selection import train_test_split

# Passo 1: One-Hot Encoding na coluna categórica 'Tipo_Ponto'
df_modelagem = pd.get_dummies(df_19_lojas, columns=['Tipo_Ponto'], drop_first=True, dtype=int)
df_modelagem = df_modelagem.set_index('ID_Loja')

print("1. Tabela após One-Hot Encoding (veja as colunas binárias no final):")
display(df_modelagem.head(6))

# Passo 2: Separando Matriz Explicativa (X) e Vetor Alvo (y)
X = df_modelagem.drop(columns=['Faturamento_Mensal_Mil_BRL'])
y = df_modelagem['Faturamento_Mensal_Mil_BRL']

# Passo 3: Separando Treino (15 lojas) e Teste (4 lojas)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=4, random_state=42)

print(f"2. Lojas de TREINO ({len(X_train)}): {list(X_train.index)}")
print(f"   Lojas de TESTE  ({len(X_test)}):  {list(X_test.index)}  <-- Reparem que a Loja_12 (que tinha Area_m2 = NaN) caiu no Teste!")

# Passo 4: Calculando a mediana APENAS nas 15 lojas de Treino (Zero Data Leakage!)
medianas_treino = X_train.median()
print("\n3. Medianas aprendidas exclusivamente no conjunto de Treino:")
print(medianas_treino[['Area_m2', 'Fluxo_Diario_Pessoas', 'Investimento_Mkt_Mil_BRL']])

# Aplicando a imputação no Treino e no Teste
X_train_prep = X_train.fillna(medianas_treino)
X_test_prep = X_test.fillna(medianas_treino)

print("\n4. Conferindo como ficaram as 4 lojas que tinham NaN (Loja_05, Loja_09, Loja_12 e Loja_14):")
display(pd.concat([X_train_prep, X_test_prep]).loc[['Loja_05', 'Loja_09', 'Loja_12', 'Loja_14']])


---
# 📈 MÓDULO 3: Regressão Linear Simples e Múltipla Passo a Passo

Agora nossas matrizes `X_train_prep` (15 lojas) e `X_test_prep` (4 lojas) estão prontas. Vamos construir o aprendizado preditivo em dois estágios: primeiro a **Regressão Linear Simples** (para enxergar a geometria dos resíduos no gráfico 2D) e depois a **Regressão Linear Múltipla**.

## 3.1. Estágio 1: Regressão Linear Simples ($\text{Area\_m2} \to \text{Faturamento}$)
Vamos treinar o modelo usando apenas a área física da loja (`Area_m2`) e desenhar na tela a distância vertical (**Resíduo $e_i = y_i - \hat{y}_i$**) de cada loja até a reta!


In [ ]:
# Selecionando apenas 1 variável explicativa (Area_m2)
X_train_simples = X_train_prep[['Area_m2']]
X_test_simples = X_test_prep[['Area_m2']]

# Treinando (.fit) nas 15 lojas de Treino
reg_simples = LinearRegression()
reg_simples.fit(X_train_simples, y_train)

b0_s = reg_simples.intercept_
b1_s = reg_simples.coef_[0]

print("======================================================================")
print(f"EQUAÇÃO DA RETA (REGRESSÃO SIMPLES):")
print(f"Faturamento_Previsto = {b0_s:.2f} + ({b1_s:.2f} * Area_m2)")
print("======================================================================")

# Visualizando geometricamente a Reta e os Resíduos (barras verticais tracejadas) nas 15 lojas de Treino!
y_train_pred_s = reg_simples.predict(X_train_simples)

plt.figure(figsize=(11, 5.5))
plt.plot(X_train_simples['Area_m2'], y_train_pred_s, color='royalblue', lw=2.5, label=f'Reta Ajustada (y_hat = {b0_s:.1f} + {b1_s:.2f}*Area)')
plt.scatter(X_train_simples['Area_m2'], y_train, color='navy', s=75, zorder=5, label='Valor Real Observado (y_i)')

# Desenhando a linha vertical do Resíduo (e_i = y_i - y_hat_i) para cada uma das 15 lojas de treino
for idx_loja, area_val, y_real_val, y_hat_val in zip(X_train_simples.index, X_train_simples['Area_m2'], y_train, y_train_pred_s):
    plt.vlines(x=area_val, ymin=min(y_real_val, y_hat_val), ymax=max(y_real_val, y_hat_val), color='crimson', linestyle=':', lw=1.8)
    plt.annotate(idx_loja.replace('Loja_', 'L'), (area_val + 0.8, y_real_val), fontsize=8.5, fontweight='bold')

plt.vlines(x=[], ymin=[], ymax=[], color='crimson', linestyle=':', lw=1.8, label='Resíduo e_i = y_i - y_hat_i (O que o OLS minimiza!)')
plt.title('Geometria dos Mínimos Quadrados: A Reta que minimiza a soma dos quadrados das linhas vermelhas')
plt.xlabel('Área da Loja (m²)')
plt.ylabel('Faturamento Mensal (R$ Mil)')
plt.legend()
plt.show()


### Analisando os Resíduos no Gráfico Acima:
Reparem nas linhas tracejadas vermelhas!  
* As lojas **`L03`, `L07`, `L15`, `L17`** ficaram sistematicamente **acima** da reta azul (resíduos positivos grandes). Por quê? Porque todas elas ficam em **Shopping** e investem mais em marketing!
* Já as lojas **`L04`, `L05`, `L08`, `L09`** ficaram **abaixo** da reta azul (resíduos negativos). Por quê? Porque ficam em **Galeria**!
* Ou seja: apenas a `Area_m2` não consegue separar o efeito da localização.

Vamos auditar numericamente o que acontece quando usamos essa **Regressão Simples** para prever as **4 Lojas Inéditas de Teste (`Loja_01`, `Loja_06`, `Loja_12`, `Loja_02`)** e entender de onde vêm as métricas **MAE**, **MSE**, **RMSE** e **$R^2$**:


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Prevendo as 4 lojas do conjunto de Teste com a Regressão Simples
y_test_pred_s = reg_simples.predict(X_test_simples)

tabela_teste_simples = pd.DataFrame({
    'Area_m2': X_test_simples['Area_m2'],
    'Real (y)': y_test,
    'Previsto_Simples (y_hat)': y_test_pred_s.round(2),
    'Residuo (e = y - y_hat)': (y_test - y_test_pred_s).round(2),
    'Erro_Absoluto |e|': np.abs(y_test - y_test_pred_s).round(2),
    'Erro_Quadratico (e²)': ((y_test - y_test_pred_s)**2).round(2)
})

print("📋 AUDITORIA PASSO A PASSO NAS 4 LOJAS DE TESTE (REGRESSÃO SIMPLES):")
display(tabela_teste_simples)

mae_s = mean_absolute_error(y_test, y_test_pred_s)
mse_s = mean_squared_error(y_test, y_test_pred_s)
rmse_s = np.sqrt(mse_s)
r2_s = r2_score(y_test, y_test_pred_s)

print(f"Como o Python calcula cada métrica a partir da tabela acima:")
print(f"1. MAE  (Média da coluna 'Erro_Absoluto |e|'):    (13.42 + 0.67 + 61.76 + 18.35) / 4 = R$ {mae_s:.2f} mil")
print(f"2. MSE  (Média da coluna 'Erro_Quadratico (e²)'): (180.23 + 0.45 + 3814.91 + 336.68) / 4 = {mse_s:.2f}")
print(f"3. RMSE (Raiz Quadrada do MSE = sqrt({mse_s:.2f})): R$ {rmse_s:.2f} mil")
print(f"4. R²   (Proporção da variância explicada):       {r2_s:.4f} ({r2_s*100:.1f}%)")


---
## 3.2. Estágio 2: Regressão Linear Múltipla (Incorporando todas as variáveis explicativas)
Notem na tabela acima que a Regressão Simples errou feio na **`Loja_12`** (previu R$ 243,24 mil, mas ela faturou R$ 305,00 mil — um erro de **R$ 61,76 mil**!).  
Agora vamos treinar a **Regressão Linear Múltipla** fornecendo todas as 5 colunas (`Area_m2`, `Fluxo_Diario_Pessoas`, `Investimento_Mkt_Mil_BRL`, `Tipo_Ponto_Rua` e `Tipo_Ponto_Shopping`) e ver a mágica acontecer:


In [ ]:
# Treinando a Regressão Linear Múltipla nas 15 lojas de Treino
reg_multipla = LinearRegression()
reg_multipla.fit(X_train_prep, y_train)

print(f"Intercepto (Beta_0 - Base Galeria): R$ {reg_multipla.intercept_:.2f} mil")
df_pesos = pd.DataFrame({
    'Variavel (Feature)': X_train_prep.columns,
    'Peso (Beta_i)': reg_multipla.coef_.round(3),
    'Interpretacao Pratica (Ceteris Paribus)': [
        'Cada +1 m² de área adiciona R$ 1,58 mil ao faturamento mensal',
        'Cada +100 pessoas/dia na calçada adiciona R$ 2,70 mil ao faturamento',
        'Cada +R$ 1 mil investido em Marketing retorna R$ 6,49 mil em faturamento',
        'Estar na RUA rende +R$ 31,96 mil em relação a uma loja de Galeria',
        'Estar em SHOPPING rende +R$ 89,77 mil em relação a uma loja de Galeria'
    ]
})
display(df_pesos)

# Prevendo as mesmas 4 lojas de Teste com a Regressão Múltipla
y_test_pred_m = reg_multipla.predict(X_test_prep)

comparativo_teste = pd.DataFrame({
    'Faturamento_Real (y)': y_test,
    'Previsto_Simples': y_test_pred_s.round(2),
    'Previsto_Multipla': y_test_pred_m.round(2),
    'Erro_Abs_Simples': np.abs(y_test - y_test_pred_s).round(2),
    'Erro_Abs_Multipla': np.abs(y_test - y_test_pred_m).round(2)
})
print("\n🔬 COMPARAÇÃO DIRETA NAS 4 LOJAS DE TESTE: REGRESSÃO SIMPLES vs. MÚLTIPLA")
display(comparativo_teste)

mae_m = mean_absolute_error(y_test, y_test_pred_m)
rmse_m = np.sqrt(mean_squared_error(y_test, y_test_pred_m))
r2_m = r2_score(y_test, y_test_pred_m)

print("==========================================================================")
print("SALTO DE QUALIDADE DO MODELO MÚLTIPLO:")
print(f" * Erro na Loja_12: caiu de R$ 61,76 mil  ---> para apenas R$ 2,54 mil!")
print(f" * MAE Médio:       caiu de R$ {mae_s:.2f} mil  ---> para R$ {mae_m:.2f} mil!")
print(f" * RMSE:            caiu de R$ {rmse_s:.2f} mil  ---> para R$ {rmse_m:.2f} mil!")
print(f" * R² no Teste:     subiu de {r2_s*100:.1f}%       ---> para {r2_m*100:.1f}%!")
print("==========================================================================")


In [ ]:
# Simulando uma tomada de decisão executiva ao vivo:
# "Professor, queremos abrir uma Loja_21 num Shopping, com 65 m², fluxo de 1.200 pessoas/dia e R$ 7 mil de marketing. Quanto ela vai faturar?"
nova_loja_21 = pd.DataFrame({
    'Area_m2': [65.0],
    'Fluxo_Diario_Pessoas': [1200.0],
    'Investimento_Mkt_Mil_BRL': [7.0],
    'Tipo_Ponto_Rua': [0],
    'Tipo_Ponto_Shopping': [1]
}, index=['Nova_Loja_21'])

previsao_loja_21 = reg_multipla.predict(nova_loja_21)[0]
print(f"🔮 PREVISÃO DO MODELO PARA A NOVA LOJA 21: R$ {previsao_loja_21:.2f} mil / mês")


---
# ☕ INTERVALO (15 Minutos)
Na volta do intervalo:
1. **Módulo 4:** Governança Oficial do Projeto do 2º Bimestre (Alinhamento Pós-Aula 09, Checkpoints, Multiplicador $1,2\times$, Avaliação por Pares de Soma Zero e Chamada dos Grupos).
2. **Módulo 5:** Laboratório de Fixação Acumulada (Aplicando **Aula 09 + Aula 10** na Base Própria ou no Dataset de Fallback `dataset_fallback_logitech_frete.csv`).


---
# 🚀 MÓDULO 4: Governança do Projeto do 2º Bimestre & Formação de Grupos

## 4.1. Nota de Alinhamento Pós-Aula 09
> Os frameworks de resolução de problemas em Ciência de Dados e a introdução ao Projeto do 2º Bimestre foram ministrados na **Aula 09 (semana passada)** e encontram-se publicados no Blackboard.  
> Em respeito aos alunos que compareceram e às normas acadêmicas, a ausência eventual na aula anterior não posterga prazos nem isenta ninguém das regras — felizmente, o Blackboard funciona 24 horas por dia e mantém todo o material acessível. Vamos consolidar abaixo o regulamento completo e registrar hoje todas as equipes e temas.

---

## 4.2. A Regra da Defasagem de 1 Semana e o Fim dos "80% de Atividades Avulsas"
* **Pontuação do 1º Bimestre encerrada:** A Prova 1 valeu **$5,0$ pontos**, e quem entregou pelo menos 6 das ~7–8 atividades práticas do 1º Bimestre garantiu **$+1,0$ ponto de bônus** (chegando a até $6,0$ pontos no 1º Bimestre).
* **No 2º Bimestre, toda atividade prática constrói o Projeto:** Não haverá mais listas soltas desconectadas. A segunda parte de cada aula será usada para aplicar no dataset do grupo o conhecimento acumulado da **semana anterior e da semana atual** (defasagem planejada de 1 semana).
* **Equipes:** De **3 a 4 alunos**, registradas obrigatoriamente na aula de hoje (**Aula 10**).

---

## 4.3. Critérios de Aceite do Tema e Obrigatoriedade Pedagógica

### 1. O que qualifica um Problema de Dados Válido? (Frameworks da Aula 09)
Não serão aceitos problemas subjetivos, triviais ou que não possam ser respondidos por algoritmos de dados (ex: *"qual cor de logotipo o dono da empresa prefere"*).
* **Alinhamento ao Negócio:** Deve existir uma pergunta clara de tomada de decisão (prever demanda, estimar custo operacional, detectar fraude/inadimplência, estimar *churn*, precificar produtos, etc.).
* **Riqueza Híbrida da Base:** O dataset precisa conter **variáveis numéricas contínuas** (para Regressão Linear e Clustering) e **variáveis categóricas/binárias** (para Classificação, Ensembles e Redes Neurais).

### 2. Obrigação Pedagógica (Fixação) vs. Adoção no Modelo Final
* **Pergunta importante:** *"Professor, o problema central do meu grupo é de Classificação (ex: aprovar ou reprovar crédito). Somos obrigados a fazer Regressão Linear?"*
* **Resposta:** **SIM.** Para fins de **fixação de conteúdo**, todos os grupos devem aplicar todas as técnicas vistas em aula no seu dataset durante os Checkpoints:
  * **Cenário A (Variável Alvo Auxiliar):** O grupo aplica a Regressão Linear para prever uma variável numérica contínua da mesma base (ex: prever a *Renda Estimada* ou o *Limite de Crédito* antes de classificar a inadimplência).
  * **Cenário B (Baseline e Descarte Justificado):** O grupo aplica o modelo linear sobre os dados, calcula as métricas antes da decisão final e documenta tecnicamente por que o descartou em favor do modelo de Classificação na entrega final. **O código precisa constar obrigatoriamente no Checkpoint.**

---

## 4.4. Cronograma Oficial de Entregas (Da Aula 10 até o Encerramento)

| Etapa | Quando? | Pontuação Base | Entregável Obrigatório no Blackboard |
| :--- | :--- | :--- | :--- |
| **Definição em Sala** | **Hoje (Aula 10)** | *Setup* | Registro dos integrantes (3–4 alunos) + Prática de Fixação (Aula 09 + Aula 10). |
| **Entrega 0 (Gatekeeper)** | **Semana que vem (Aula 11)** | **Obrigatório** *(Trava sem nota)* | **Escopo Validado + Dataset do Grupo** já com o *Data Wrangling* (Aula 09) e a *Regressão Linear* (Aula 10) aplicados no Notebook. |
| **Checkpoint 1** | **Aula 12 / 13** | **1,25 pts** *(Desenvolvimento)* | Notebook evoluído com **Wrangling + Regressão Linear + Clustering (Aprendizado Não Supervisionado)** aplicados e interpretados. |
| **Checkpoint 2** | **Aula 14** | **1,25 pts** *(Desenvolvimento)* | Notebook evoluído com **Classificação, Ensembles e Redes Neurais** rodando com métricas comparativas. |
| **Última Aula** | **Última Aula do Semestre** | *Mentoria Geral* | **Sessão Geral de Tira-Dúvidas.** Em vez de apresentações apressadas em sala, vocês terão mentoria para aplicar inclusive o conteúdo da penúltima aula. |
| **ENTREGA FINAL** | **+6 dias corridos após a Última Aula** | **2,50 pts** *(Apresentação)* | **Vídeo Pitch + Notebook Definitivo no Blackboard + Avaliação por Pares.** *(Corte seco e improrrogável para garantir tempo hábil de correção antes da Prova Final).* |

> ⚖️ **Direito de Revisão na Prova Final:** Todos os grupos entregam o vídeo até o 6º dia após a última aula. Após a correção, quem ficar de Exame Final ou discordar fundamentadamente da nota poderá comparecer no **dia oficial da Prova Final** para discutir e defender o trabalho presencialmente comigo.


---
## 4.5. A Matemática da Nota: Multiplicador de Mundo Real ($1,2\times$) e Avaliação por Pares de Soma Zero

O Projeto Semestral vale **$5,0$ pontos de base** ($2,5$ pontos dos Checkpoints de desenvolvimento + $2,5$ pontos da Entrega Final em Vídeo/Notebook). Sobre essa base aplicam-se dois multiplicadores:

### 1. Multiplicador de Mundo Real ($M_{real}$)
* **Bases Públicas Tradicionais (Kaggle, UCI, Governo) ou Sintéticas:** Multiplicador $M_{real} = \mathbf{1,0\times}$ (Máximo de $5,0$ pontos).
* **Dados Operacionais Reais de Negócio (extraídos da empresa onde trabalham, negócio real ou base corporativa complexa):** Multiplicador $M_{real} = \mathbf{1,2\times}$ (Pode adicionar até **$+1,0$ ponto extra**, elevando o projeto para **$6,0$ pontos**!).
* *(Nota: Somando até $6,0$ pontos do 1º Bimestre + até $6,0$ pontos do 2º Bimestre, a soma teórica chega a $12,0$, sendo cortada automaticamente em **$10,0$** no sistema).*

### 2. Multiplicador de Mérito Individual — Avaliação por Pares de Soma Zero ($F_{ind}$)
Para evitar que todo mundo dê nota máxima ao colega por camaradagem, utilizaremos um **sistema de orçamento fixo (Soma Zero)**:
* Em um grupo de $N$ alunos, cada integrante possui exatamente **$5 \times (N - 1)$ pontos** para distribuir entre seus $N - 1$ colegas (não existe autoavaliação).
  * **Exemplo (Grupo de 4 pessoas):** Cada aluno tem **15 pontos** para dividir entre os seus 3 colegas.
  * Se todos trabalharam em igualdade: distribui **5, 5 e 5**.
  * Se um colega não fez nada e outros dois carregaram o trabalho dele: o ausente recebe **0**, e os pontos dele migram para quem trabalhou mais (ex: **0, 7 e 8**).
* Calculada a **Média de Pontos Recebidos ($M_i$)** pelo aluno $i$, o seu Fator Individual ($F_{ind}$) é dado por:

$$F_{ind} = 0,5 + 0,1 \times M_i$$

$$\mathbf{Nota\_Final_{2B}} = \text{Nota\_Base} \times M_{real} \times F_{ind}$$

* **Quem leva $0$ dos colegas ($M_i = 0$):** Fica com multiplicador **$0,5\times$** (perde metade da nota do grupo).
* **Quem leva $5$ dos colegas ($M_i = 5$):** Fica com multiplicador **$1,0\times$** (recebe $100\%$ da nota do grupo).
* **Quem absorve os pontos do colega ausente (ex: $M_i = 7,5$):** Fica com multiplicador **$1,25\times$** (**ganha $+25\%$ de bônus sobre a nota do grupo!**).


In [ ]:
# Simulador Oficial da Nota do 2º Bimestre (Rode para testar cenários!)
def simular_avaliacao_pares(nota_base_grupo, dados_reais_negocio, notas_recebidas_dict):
    m_real = 1.2 if dados_reais_negocio else 1.0
    linhas = []
    for aluno, lista_notas in notas_recebidas_dict.items():
        media_i = np.mean(lista_notas)
        f_ind = 0.5 + 0.1 * media_i
        nota_final = nota_base_grupo * m_real * f_ind
        linhas.append({
            'Integrante': aluno,
            'Pontos Recebidos dos Colegas': str(lista_notas),
            'Média (M_i)': round(media_i, 2),
            'Fator Individual (F_ind)': f"{f_ind:.2f}x",
            'Nota Grupo x M_real': round(nota_base_grupo * m_real, 2),
            'NOTA INDIVIDUAL 2º BIM': round(nota_final, 2)
        })
    return pd.DataFrame(linhas)

print("Exemplo: Grupo tirou 4.5 de Nota Base (Dados Públicos = 1.0x), mas 'Integrante_D' não trabalhou:")
exemplo_soma_zero = {
    'Integrante_A': [8, 7, 5],
    'Integrante_B': [7, 8, 5],
    'Integrante_C': [8, 7, 5],
    'Integrante_D (Carona)': [0, 0, 0]
}
display(simular_avaliacao_pares(4.5, False, exemplo_soma_zero))


In [ ]:
# ==============================================================================
# REGISTRO OFICIAL DO GRUPO E TEMA (PREENCHER AGORA NA AULA 10)
# ==============================================================================
grupo_integrantes = [
    "1. Nome Completo",
    "2. Nome Completo",
    "3. Nome Completo",
    # "4. Nome Completo (Opcional)"
]

cenario_negocio = "Qual problema/decisão de negócio o grupo vai resolver?"
origem_dataset = "Dados Reais de Empresa (1.2x) ou Base Pública (1.0x)?"

print("Equipe:", grupo_integrantes)
print("Problema:", cenario_negocio)
print("Fonte:", origem_dataset)


---
# 🛠️ MÓDULO 5: Laboratório de Fixação da Aula 10 (Consolidando Aula 09 + Aula 10)

Seguindo a regra de **defasagem de 1 semana**, nos próximos 40 minutos vocês aplicarão o pipeline acumulado da **Aula 09 (*Data Wrangling*)** + **Aula 10 (*Preparação Preditiva e Regressão Linear*)**:

* **CENÁRIO 1 (Grupos que JÁ trouxeram dataset próprio hoje):** Carreguem a base do grupo e executem as 5 etapas abaixo diretamente nos dados do projeto (isso já adianta a entrega obrigatória da **Aula 11**, semana que vem!).
* **CENÁRIO 2 (Grupos que AINDA NÃO trouxeram dataset próprio hoje):** Vocês executarão as 5 etapas abaixo utilizando o **Dataset de Fallback Oficial (`dataset_fallback_logitech_frete.csv`)**, que já está no Blackboard e também é carregado/gerado automaticamente na célula a seguir.

---

### 📦 Sobre o Dataset de Fallback (`dataset_fallback_logitech_frete.csv` — 255 Pedidos):
A **LogiTech Brasil** quer um modelo de Regressão Linear Múltipla capaz de prever o **`Custo_Operacional_Entrega_BRL` (Alvo $y$)** de cada frete a partir de:
* `Regiao_Destino` (Categórica: Sul, Sudeste, Nordeste, Centro-Oeste, Norte).
* `Modalidade_Envio` (Categórica suja: mistura `'Padrao'`, `'PADRAO'`, `'  expresso '`, `'economico '`).
* `Distancia_KM` (Numérica contínua — contém valores `NaN`).
* `Peso_KG` (Numérica contínua — contém valores `NaN` e erros de domínio com `-50.0 kg`).
* `Tempo_Separacao_Min` (Numérica contínua — tempo de galpão).
* `Valor_Mercadoria_BRL` (Numérica contínua — valor da nota fiscal).
* **Atenção:** A base contém **5 linhas duplicadas** no final e **3 outliers extremos** onde `Custo_Operacional_Entrega_BRL` foi digitado como `29999.00`!


In [ ]:
import os

csv_fallback = 'dataset_fallback_logitech_frete.csv'

# Carrega o CSV baixado do Blackboard ou gera automaticamente caso não esteja na pasta atual
if not os.path.exists(csv_fallback):
    np.random.seed(2026)
    n_p = 250
    dist_km = np.random.gamma(shape=3.0, scale=110.0, size=n_p).round(1)
    p_kg = np.random.uniform(1.0, 40.0, size=n_p).round(2)
    val_merc = (p_kg * np.random.uniform(20, 75, size=n_p) + np.random.normal(120, 25, size=n_p)).round(2)
    t_sep = np.random.uniform(10, 90, size=n_p).round(1)
    mod = np.random.choice(['Padrao', 'Expresso', 'Economico', '  expresso ', 'PADRAO', 'economico '], size=n_p, p=[0.40, 0.25, 0.20, 0.05, 0.05, 0.05])
    reg = np.random.choice(['Sul', 'Sudeste', 'Nordeste', 'Centro-Oeste', 'Norte'], size=n_p, p=[0.25, 0.40, 0.20, 0.10, 0.05])
    b_exp = np.where(pd.Series(mod).str.strip().str.lower() == 'expresso', 38.0, 0.0)
    c_ent = (15.0 + 0.09 * dist_km + 2.10 * p_kg + 0.015 * val_merc + 0.12 * t_sep + b_exp + np.random.normal(0, 6.5, size=n_p)).round(2)
    df_tmp = pd.DataFrame({
        'ID_Pedido': [f'PED-{1001+i}' for i in range(n_p)],
        'Regiao_Destino': reg, 'Modalidade_Envio': mod, 'Distancia_KM': dist_km,
        'Peso_KG': p_kg, 'Tempo_Separacao_Min': t_sep, 'Valor_Mercadoria_BRL': val_merc,
        'Custo_Operacional_Entrega_BRL': c_ent
    })
    df_tmp.loc[[4, 18, 42, 77, 105, 140, 188, 210, 235], 'Distancia_KM'] = np.nan
    df_tmp.loc[[9, 33, 68, 112, 155, 199, 222], 'Peso_KG'] = np.nan
    df_tmp.loc[[15, 89, 174], 'Peso_KG'] = -50.0
    df_tmp.loc[[25, 130, 205], 'Custo_Operacional_Entrega_BRL'] = 29999.00
    df_tmp = pd.concat([df_tmp, df_tmp.iloc[[0, 1, 2, 10, 20]]], ignore_index=True)
    df_tmp.to_csv(csv_fallback, index=False, encoding='utf-8')

df_fixacao = pd.read_csv(csv_fallback)
print(f"✅ Dataset '{csv_fallback}' carregado! Shape: {df_fixacao.shape}")
display(df_fixacao.head(10))


### 🎯 Checklist de Execução para o Grupo (5 Passos):
1. **Passo 1 — Diagnóstico (Aula 09):** Rastreie quantas duplicatas (`.duplicated().sum()`), valores nulos (`.isnull().sum()`), categorias quebradas em `Modalidade_Envio` (`.value_counts()`) e anomalias numéricas (`.describe()`) existem na tabela.
2. **Passo 2 — Limpeza e Filtragem de Outliers (Aula 09):**
   * Remova as duplicatas (`.drop_duplicates(keep='first')`).
   * Padronize `Modalidade_Envio` com `.str.strip().str.title()`.
   * Converta `Peso_KG < 0` em `np.nan` e remova os registros onde `Custo_Operacional_Entrega_BRL > 5000`.
3. **Passo 3 —Preparação Preditiva Livre de *Data Leakage* (Aula 10):**
   * Aplique `pd.get_dummies(..., drop_first=True, dtype=int)` em `Regiao_Destino` e `Modalidade_Envio`.
   * Defina `X` (sem `ID_Pedido` e sem a coluna alvo) e `y` (`Custo_Operacional_Entrega_BRL`).
   * Faça o `train_test_split(X, y, test_size=0.2, random_state=42)` e preencha os `NaN` usando a **mediana calculada apenas no `X_train`**.
4. **Passo 4 — Regressão Linear Simples vs. Múltipla (Aula 10):**
   * Treine uma **Regressão Simples** usando apenas `Distancia_KM` e meça o $MAE$, $RMSE$ e $R^2$ no Teste.
   * Treine uma **Regressão Múltipla** usando todas as variáveis explicativas e compare a evolução do $MAE$, $RMSE$ e $R^2$ no Teste!
5. **Passo 5 — Interpretação de Negócio (Aula 10):**
   * Olhando os pesos $\beta_i$ do Modelo Múltiplo, responda em texto: **Quanto custa a mais (em R$) cada 1 Kg extra de peso? E qual é o acréscimo médio de custo da modalidade `Expresso`?**


In [ ]:
# ==============================================================================
# ESPAÇO DE RESOLUÇÃO DO GRUPO — PASSOS 1 E 2 (DIAGNÓSTICO E WRANGLING - AULA 09)
# ==============================================================================


In [1]:
# ==============================================================================
# ESPAÇO DE RESOLUÇÃO DO GRUPO — PASSO 3 (ENCODING, SPLIT E IMPUTAÇÃO - AULA 10)
# ==============================================================================# ==============================================================================
# ESPAÇO DE RESOLUÇÃO DO GRUPO — PASSOS 4 E 5 (REGRESSÃO E MÉTRICAS - AULA 10)
# ==============================================================================
